In [ ]:
import geopandas as gpd
from src.utils.paths import EXTERNAL_DATA_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR

rename_vars = {
    "B19113EST1": "avg_hh_income",
}

boundaries = gpd.read_file(EXTERNAL_DATA_DIR / "NYC Borough Boundaries" / "Borough_Boundaries_20260131.geojson")
buildings = gpd.read_file(RAW_DATA_DIR / "USBuildingFootprints" / "NewYork.geojson" / "NewYork.geojson")
sae = gpd.read_file(RAW_DATA_DIR / "ACS 5YR Small Area Estimates" / "ACS_5YR_ESTIMATES_SOCIOECONOMIC_TRACT_-5061620031031943076.zip")\
        .to_crs("EPSG:4326")\
        .rename(columns=rename_vars)\
        [['GEOID', 'STATE', 'COUNTY', 'TRACT', 'NAME', 'CNTY_FIPS', 'geometry'] + list(rename_vars.values())]

In [42]:
#  Box -74.443359,40.269048,-73.100281,41.213788
buildings = buildings.cx[-74.443359:-73.100281, 40.269048:41.213788]
sae = sae.cx[-74.443359:-73.100281, 40.269048:41.213788]

In [ ]:
nyc_buildings = buildings.drop(columns=["release"])\
    .sjoin(boundaries[["boroname", "geometry"]], how="inner")\
    .drop(columns=["index_right"])\
    .sjoin(sae, how="inner")      
nyc_buildings.to_parquet(PROCESSED_DATA_DIR / "nyc_buildings_with_sae.parquet")

In [16]:
# Testing

import geopandas as gpd
from src.utils.paths import EXTERNAL_DATA_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR

nyc_buildings = gpd.read_parquet(PROCESSED_DATA_DIR / "nyc_buildings_with_sae.parquet")
nyc_buildings

,capture_da,boroname,index_righ,GEOID,STATE,COUNTY,TRACT,NAME,CNTY_FIPS,avg_hh_inc,geometry
0,None,Queens,74993,36081157901,36,081,157901,Census Tract 1579.01,36081,122969,"POLYGON ((-73.70119 40.74033, -73.70122 40.740..."
1,None,Queens,74993,36081157901,36,081,157901,Census Tract 1579.01,36081,122969,"POLYGON ((-73.70137 40.74804, -73.70124 40.748..."
2,None,Queens,74993,36081157901,36,081,157901,Census Tract 1579.01,36081,122969,"POLYGON ((-73.7023 40.74155, -73.70216 40.7415..."
3,None,Queens,74993,36081157901,36,081,157901,Census Tract 1579.01,36081,122969,"POLYGON ((-73.70257 40.74112, -73.70244 40.741..."
4,None,Queens,74995,36081157903,36,081,157903,Census Tract 1579.03,36081,104135,"POLYGON ((-73.70276 40.73746, -73.70283 40.737..."
...,...,...,...,...,...,...,...,...,...,...,...
457302,7/25/2019-7/30/2019,Staten Island,64529,36085024401,36,085,024401,Census Tract 244.01,36085,127250,"POLYGON ((-74.25161 40.50454, -74.25176 40.504..."
457303,7/25/2019-7/30/2019,Staten Island,64529,36085024401,36,085,024401,Census Tract 244.01,36085,127250,"POLYGON ((-74.2518 40.50749, -74.25167 40.5075..."
457304,7/25/2019-7/30/2019,Staten Island,64529,36085024401,36,085,024401,Census Tract 244.01,36085,127250,"POLYGON ((-74.25189 40.50679, -74.2519 40.5067..."
457305,7/25/2019-7/30/2019,Staten Island,64529,36085024401,36,085,024401,Census Tract 244.01,36085,127250,"POLYGON ((-74.2529 40.507, -74.2529 40.50706, ..."
